# 🥇 Capa Gold — Construcción de Features RFM
**Flujo:** `bronze_sales_raw` → `silver_sales_cleaned` → **`gold_customer_features`**

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
print('✅ Librerías importadas correctamente')

In [ ]:
df_silver = spark.table('bigdata.default.silver_sales_cleaned')
print(f'📥 Registros desde Silver: {df_silver.count():,}')
df_silver.printSchema()

In [ ]:
snapshot_date_str = (
    df_silver
    .agg(F.max('fecha_factura').alias('max_date'))
    .withColumn('snapshot_date', F.date_add(F.col('max_date'), 1))
    .collect()[0]['snapshot_date']
)
print(f'📅 Snapshot date: {snapshot_date_str}')

In [ ]:
snapshot_lit = F.lit(snapshot_date_str)

df_rfm = (
    df_silver
    .groupBy('id_cliente')
    .agg(
        F.datediff(snapshot_lit, F.max('fecha_factura')).cast(IntegerType()).alias('recency'),
        F.countDistinct('numero_factura').cast(IntegerType()).alias('frequency'),
        F.round(F.sum('TotalAmount'), 2).cast(DoubleType()).alias('monetary'),
        F.max('fecha_factura').alias('ultima_compra'),
        F.first('pais').alias('pais')
    )
)

print(f'👥 Clientes únicos: {df_rfm.count():,}')
df_rfm.display()

In [ ]:
print('📊 Estadísticas RFM:')
df_rfm.select('recency', 'frequency', 'monetary').describe().display()

In [ ]:
df_gold = df_rfm.withColumn('gold_timestamp', F.current_timestamp())

df_gold.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('bigdata.default.gold_customer_features')

print('✅ Tabla gold_customer_features guardada!')

In [ ]:
bronze = spark.table('bigdata.default.bronze_sales_raw').count()
silver = spark.table('bigdata.default.silver_sales_cleaned').count()
gold   = spark.table('bigdata.default.gold_customer_features').count()

print('=' * 50)
print('       RESUMEN PIPELINE MEDALLÓN')
print('=' * 50)
print(f'  🥉 Bronze : {bronze:>10,} transacciones')
print(f'  🥈 Silver : {silver:>10,} transacciones limpias')
print(f'  🥇 Gold   : {gold:>10,} clientes con RFM')
print('=' * 50)
print('✅ Listo para el equipo de ML → Notebook 04')